# Web Scraping with Requests, BeautifulSoup, and Pandas

## Overview
This notebook practices inspecting HTML and extracting tabular data from public webpages. It first explores the HTML on Scrapethissite, then collects a table from Wikipedia, loads it into a Pandas DataFrame, saves it to CSV, and checks and cleans the data.

## Tools
- Python
- Requests
- BeautifulSoup
- Pandas
- Jupyter Notebook

## Setup
Install the required packages if needed:

`pip install requests beautifulsoup4 pandas`

Run the notebook from top to bottom. The CSV output is saved as `largest_companies.csv` in the current working directory.

## 1. Import libraries

In [ ]:
from pathlib import Path

import pandas as pd
import requests
from bs4 import BeautifulSoup

## 2. Inspect a practice webpage
Use Scrapethissite to practice finding HTML elements and extracting their text.

In [ ]:
practice_url = "https://www.scrapethissite.com/pages/forms/"
response = requests.get(practice_url, timeout=30)
response.raise_for_status()

soup = BeautifulSoup(response.text, "html.parser")

### Inspect the HTML structure

In [ ]:
print(soup.prettify()[:5000])

### Find elements and extract text

In [ ]:
first_div = soup.find("div")
matching_divs = soup.find_all("div", class_="col-md-12")
lead_paragraphs = soup.find_all("p", class_="lead")

first_lead_text = (
    lead_paragraphs[0].get_text(strip=True)
    if lead_paragraphs
    else None
)

print("First div found:", first_div is not None)
print("Matching divs:", len(matching_divs))
print("Lead paragraphs:", lead_paragraphs)
print("First lead paragraph:", first_lead_text)

In [ ]:
table_headers = soup.find_all("th")
first_header_text = table_headers[0].get_text(strip=True) if table_headers else None
print("Table headers found:", len(table_headers))
print("First header:", first_header_text)

## 3. Scrape a table from Wikipedia
This section extracts the first table from the selected public Wikipedia page. Website HTML can change, so the table selection should be checked if the page layout changes.

In [ ]:
companies_url = (
    "https://en.wikipedia.org/wiki/"
    "List_of_largest_companies_in_the_United_States_by_revenue"
)
headers = {"User-Agent": "LearningWebScraping/1.0 (educational project)"}

response = requests.get(companies_url, headers=headers, timeout=30)
response.raise_for_status()
companies_soup = BeautifulSoup(response.text, "html.parser")

print("HTTP status:", response.status_code)
print("Final URL:", response.url)

### Locate the table

In [ ]:
tables = companies_soup.find_all("table", class_="wikitable")
if not tables:
    raise ValueError("No wikitable was found. The webpage structure may have changed.")

table = tables[0]
print("Tables found:", len(tables))

### Extract table headers and rows

In [ ]:
table_headers = [header.get_text(" ", strip=True) for header in table.find_all("th")]
print(table_headers)

rows = []
for row in table.find_all("tr"):
    cells_in_row = row.find_all("td")
    if not cells_in_row:
        continue

    values = [cell.get_text(" ", strip=True) for cell in cells_in_row]
    if len(values) == len(table_headers):
        rows.append(values)

companies_df = pd.DataFrame(rows, columns=table_headers)
companies_df.head()

## 4. Save the scraped data to CSV

In [ ]:
output_path = Path("largest_companies.csv")
companies_df.to_csv(output_path, index=False)
print(f"Saved {len(companies_df)} rows to {output_path.resolve()}")

## 5. Check the saved dataset

In [ ]:
check_df = pd.read_csv(output_path)

print("Rows and columns:", check_df.shape)
print("Column names:", check_df.columns.tolist())
print("Missing values per column:")
print(check_df.isna().sum())
print("\nData types:")
print(check_df.dtypes)

check_df.head()

## 6. Clean numeric columns
Convert revenue, employee counts, and revenue growth from text to numeric values. If the source changes its formatting or column names, these steps may need adjustment.

In [ ]:
clean_df = check_df.copy()

revenue_column = "Revenue (USD millions)"
employees_column = "Employees"
growth_column = "Revenue growth"

required_columns = [revenue_column, employees_column, growth_column]
missing_columns = [column for column in required_columns if column not in clean_df.columns]
if missing_columns:
    raise KeyError(f"Expected columns not found: {missing_columns}")

clean_df[revenue_column] = pd.to_numeric(
    clean_df[revenue_column].astype(str).str.replace(",", "", regex=False),
    errors="coerce",
)
clean_df[employees_column] = pd.to_numeric(
    clean_df[employees_column].astype(str).str.replace(",", "", regex=False),
    errors="coerce",
)
clean_df[growth_column] = pd.to_numeric(
    clean_df[growth_column].astype(str).str.replace("%", "", regex=False),
    errors="coerce",
)

clean_df.head()

## 7. Validate the cleaned dataset

In [ ]:
print("Cleaned shape:", clean_df.shape)
print("Duplicate rows:", clean_df.duplicated().sum())
print("\nMissing values:")
print(clean_df.isna().sum())
print("\nData types:")
print(clean_df.dtypes)

## 8. Save the cleaned dataset

In [ ]:
clean_output_path = Path("largest_companies_cleaned.csv")
clean_df.to_csv(clean_output_path, index=False)
print(f"Saved cleaned data to {clean_output_path.resolve()}")

## Notes
This is an educational scraping exercise. The results reflect the public webpage at the time the notebook is run; scraped content may change. Check the target website’s terms and access rules before reusing scraped data.